# 12. Multi-Agent Orchestration (코드 기반 멀티에이전트 오케스트레이션)

**04_Handoffs**에서는 LLM이 자율적으로 핸드오프를 결정하는 **LLM 기반 오케스트레이션**을 배웠습니다.
이번에는 **코드로 직접** 에이전트 흐름을 제어하는 패턴을 학습합니다.

### LLM 기반 vs 코드 기반 오케스트레이션

| 구분 | LLM 기반 (Handoffs) | 코드 기반 |
|------|-------------------|----------|
| 제어 주체 | LLM이 자율 판단 | 개발자가 코드로 제어 |
| 예측 가능성 | 낮음 (LLM 판단에 의존) | 높음 (결정적 흐름) |
| 유연성 | 높음 | 보통 |
| 비용/속도 | 변동적 | 예측 가능 |
| 적합한 경우 | 복잡한 대화, 분류가 어려운 경우 | 파이프라인, 배치 처리, 정해진 워크플로우 |

### 주요 패턴
1. **순차 체이닝 (Sequential Chaining)**: A → B → C
2. **병렬 실행 (Parallelization)**: A, B, C 동시 실행
3. **분류 → 라우팅 (Classification + Routing)**: 분류 후 전문 에이전트에 전달

## 1. 순차 체이닝 (Sequential Chaining)

에이전트A의 출력을 에이전트B의 입력으로 전달하는 파이프라인입니다.
각 단계가 순서대로 실행됩니다.

```
[작성 에이전트] → 초안 → [편집 에이전트] → 수정본 → [번역 에이전트] → 최종 결과
```

In [ ]:
# Step 1: 글 작성 에이전트
# Step 2: 편집 에이전트
# Step 3: 번역 에이전트
# 순차 체이닝 실행
    # Step 1: 글 작성
    # Step 2: 편집 (이전 단계의 출력을 입력으로 전달)
    # Step 3: 번역 (이전 단계의 출력을 입력으로 전달)

## 2. 병렬 실행 (Parallelization)

서로 독립적인 에이전트를 `asyncio.gather()`로 **동시에** 실행하여 시간을 절약합니다.

```
        ┌→ [감성 분석 에이전트] → 결과1   ─┐
[입력] ──┼→ [키워드 추출 에이전트] → 결과2 ─┼→ [종합]
        └→ [요약 에이전트] → 결과3   ─────┘
```

In [ ]:
# 3개의 독립적인 분석 에이전트
# asyncio.gather로 3개 에이전트를 동시에 실행

## 3. 분류 → 라우팅 (Classification + Routing)

**Structured Output**으로 입력을 분류한 후, 결과에 따라 적절한 전문 에이전트를 실행합니다.

```
[입력] → [분류 에이전트] → 카테고리 → [전문 에이전트 선택] → [전문 에이전트 실행] → [응답]
```

In [ ]:
# 분류 결과를 위한 Pydantic 모델
class RequestClassification(BaseModel):
# 분류 에이전트
# 전문 에이전트들
# 카테고리 → 에이전트 매핑

In [ ]:
# 분류 → 라우팅 실행
        # Step 1: 분류
        # Step 2: 라우팅 - 분류 결과에 따라 적절한 에이전트 선택
        # Step 3: 전문 에이전트 실행

In [ ]:
# 테스트 1: 기술지원 질문

In [ ]:
# 테스트 2: 결제 질문

In [ ]:
# 테스트 3: 일반 질문

## 4. 종합 패턴: 병렬 + 순차 조합

실전에서는 여러 패턴을 **조합**하여 사용합니다.

```
[고객 리뷰] ──→ [병렬 분석] ──→ [종합 에이전트] ──→ [최종 보고서]
                 ├ 감성 분석
                 ├ 키워드 추출
                 └ 요약
```

In [ ]:
# 종합 보고서를 작성하는 에이전트
    # Step 1: 병렬 분석 (동시 실행)
    # Step 2: 순차 체이닝 - 분석 결과를 종합

### 실습 문제

**뉴스 기사 브리핑 파이프라인 (병렬 분석 + 순차 종합 패턴)**

본문 4번 패턴처럼 **병렬 분석 → 순차 종합** 구조로 뉴스 기사 브리핑 시스템을 만드세요.

1. **병렬 분석 (asyncio.gather)**: 세 에이전트를 동시에 실행하세요.
   - **제목 생성기**: 기사에 어울리는 제목 1개 생성
   - **핵심 문장 추출기**: 기사에서 가장 중요한 문장 2개 추출
   - **카테고리 분류기**: `경제 / 기술 / 사회` 중 하나로 분류

2. **순차 종합**: **브리핑 작성기** 에이전트가 세 분석 결과를 입력으로 받아
   `제목 / 카테고리 / 핵심 내용 / 한 줄 평` 형식의 **3~4줄 브리핑**을 작성하세요.

3. 전체 파이프라인을 `trace("뉴스 브리핑 파이프라인")`으로 감싸세요.

### 테스트 입력 예시

```
국내 연구진이 차세대 이차전지인 전고체 배터리의 수명을 3배 늘리는
신소재를 개발했다. 이번 기술은 전기차 주행거리 향상에 크게 기여할 것으로
기대되며, 연구팀은 3년 내 상용화를 목표로 국내 배터리 기업들과
협력을 논의 중이라고 밝혔다.
```

👉 세 분석 결과가 동시에 출력된 뒤, 이를 종합한 최종 브리핑이 출력되어야 합니다.
